# Resume Screening with LLMs

**Goal**: Build a resume scoring system using structured outputs and submit scores to the leaderboard.

## Setup

In [1]:
import json
from resume_utils import load_resumes, analyze_resume, submit_score

# --- Configuration ---
OPENROUTER_API_KEY = "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"  # Paste your key here
MODEL = "anthropic/claude-sonnet-4-6"
TEAM_NAME = "Nick"
LEADERBOARD_URL = "http://ai-leaderboard.site"

if not OPENROUTER_API_KEY.strip():
    raise RuntimeError(
        "Please set OPENROUTER_API_KEY above before running this notebook.\n"
    )

if not TEAM_NAME.strip():
    raise RuntimeError(
        "Please set your TEAM NAME.\n"
    )
    

# Load resume data
resumes = load_resumes("../data/resumes_final.csv")
sample_id = list(resumes.keys())[0]
sample_resume = resumes[sample_id]

print(f"API key configured")
print(f"Model: {MODEL}")
print(f"Loaded {len(resumes)} resumes")
print(f"Sample resume ID: {sample_id}")
print(f"Resume preview: {sample_resume['Resume_str'][:200]}...")

API key configured
Model: anthropic/claude-sonnet-4-6
Loaded 130 resumes
Sample resume ID: 10089434
Resume preview:          INFORMATION TECHNOLOGY TECHNICIAN I       Summary     Versatile Systems Administrator possessing superior troubleshooting skills for networking issues, end user problems, and network security...


## Example: Score a Resume and Submit to Leaderboard

In [2]:
# Score a resume: simple prompt with structured output
prompt = "On a scale of 1 to 100, how good of a fit is this person for a software engineering job? Return a JSON object with a 'score' field (integer 1-100) and a brief 'reasoning' field."

output_schema = '{"score": 85, "reasoning": "Strong experience with..."}'

result = analyze_resume(
    OPENROUTER_API_KEY,
    prompt,
    sample_resume["Resume_str"],
    output_schema,
    model=MODEL,
)


if result["error"]:
    print(f"Error: {result['error']}")
else:
    score = result["result"]["score"]
    reasoning = result["result"].get("reasoning", "")
    print(f"Resume {sample_id} score: {score}/100")
    print(f"Reasoning: {reasoning}")
    print(f"Tokens used: {result['usage'].get('total_tokens', 0)}")

    # Submit to leaderboard
    response = submit_score(TEAM_NAME, sample_id, score, api_url=LEADERBOARD_URL)
    print(f"\nSubmitted to leaderboard: {response}")

Resume 10089434 score: 62/100
Reasoning: This candidate has solid IT infrastructure and systems administration experience with relevant skills like Active Directory, Azure, Office 365, PowerShell, and VMware. However, the role is 'IT Technician I' rather than a software engineering position. While there are some software-adjacent activities (MVC 4/.NET training, test script development, QA planning, TFS usage), the core experience is heavily focused on systems administration, networking, and IT operations rather than software development. The candidate lacks demonstrated proficiency in core software engineering competencies such as algorithms, data structures, software design patterns, version control workflows, or substantial coding projects. The brief exposure to .NET/HTML5/CSS3 through training and limited QA scripting is insufficient to qualify as strong software engineering experience.
Tokens used: 916

Submitted to leaderboard: {'status': 'ok', 'team_name': 'Nick', 'resume_id': '